# Notebook 03 — Error Analysis & Uncertainty Exploration

Post-training analysis:
- Per-sample Dice scores and error distribution
- Uncertainty vs error scatter plots
- Calibration curves
- Worst-case / best-case qualitative review
- Bland-Altman plot for bone-loss percentage

**Requires a trained checkpoint. Run scripts/06 first.**

**IMPORTANT: Only use metrics from real DenPAR training.**

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

CHECKPOINT = '../outputs/checkpoints/multitask/best.pt'
DATA_ROOT  = '../data/raw/DenPAR'
PROC_DIR   = '../data/processed'
METRICS_DIR = '../outputs/metrics'
DEVICE     = 'cpu'   # change to 'cuda' if available

if not Path(CHECKPOINT).exists():
    print(f'[WARN] Checkpoint not found: {CHECKPOINT}')
    print('Run scripts/04 or scripts/05 to train a model first.')

In [ ]:
# Load latest metrics JSON if available
metrics_files = sorted(Path(METRICS_DIR).glob('eval_results_*.json'))
if metrics_files:
    with open(metrics_files[-1]) as f:
        metrics = json.load(f)
    print(f'Loaded metrics from: {metrics_files[-1].name}')
    for section, vals in metrics.items():
        print(f'\n  [{section.upper()}]')
        for k, v in vals.items():
            print(f'    {k}: {v:.4f}' if isinstance(v, float) else f'    {k}: {v}')
else:
    print('[INFO] No metrics JSON found yet. Run scripts/06_evaluate_testset.py first.')
    metrics = {}

In [ ]:
# Per-sample inference (requires checkpoint + data)
if Path(CHECKPOINT).exists():
    from src.data.denpar_dataset import DenPARRawDataset
    from src.models.unet_baseline import MultiTaskUNet
    from src.models.uncertainty import MCDropoutEstimator
    from src.evaluation.metrics_segmentation import dice

    model = MultiTaskUNet(in_channels=1)
    ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()

    ds = DenPARRawDataset(DATA_ROOT, 'Testing', img_size=512, max_samples=30)
    loader = DataLoader(ds, batch_size=1)
    estimator = MCDropoutEstimator(model, n_samples=10, device=DEVICE)

    results = []
    for batch in loader:
        image = batch['image']
        with torch.no_grad():
            out = model(image)
        pred = (torch.softmax(out['tooth_logits'], dim=1)[0, 1].numpy() > 0.5).astype(float)
        gt   = batch['tooth_mask'][0, 0].numpy().astype(float)
        d    = dice(pred, gt)
        mc   = estimator.predict(image, task_key='tooth_logits')
        unc  = mc['norm_entropy'][0, 0].numpy().mean()
        results.append({'image_id': batch['image_id'][0], 'dice': d, 'uncertainty': unc,
                        'image': image[0,0].numpy(), 'gt': gt, 'pred': pred,
                        'unc_map': mc['norm_entropy'][0,0].numpy()})

    print(f'Evaluated {len(results)} samples')
    print(f'Mean Dice: {np.mean([r["dice"] for r in results]):.4f}')
    print(f'Mean Uncertainty: {np.mean([r["uncertainty"] for r in results]):.4f}')
else:
    results = []
    print('[SKIP] Checkpoint not found — skipping per-sample analysis.')

In [ ]:
# Uncertainty vs Error scatter
if results:
    from scipy import stats
    unc_arr = np.array([r['uncertainty'] for r in results])
    err_arr = np.array([1 - r['dice'] for r in results])
    rho, p = stats.spearmanr(unc_arr, err_arr)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.scatter(unc_arr, err_arr, alpha=0.7, s=40, c='steelblue')
    ax.set_xlabel('Predictive Entropy (uncertainty)')
    ax.set_ylabel('Segmentation Error (1 - Dice)')
    ax.set_title(f'Uncertainty vs Error  (Spearman ρ = {rho:.3f}, p = {p:.3e})')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../outputs/figures/uncertainty_error_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Dice score distribution
if results:
    dices = [r['dice'] for r in results]
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(dices, bins=15, color='steelblue', edgecolor='white')
    ax.axvline(np.mean(dices), color='red', linestyle='--', label=f'Mean={np.mean(dices):.3f}')
    ax.set_xlabel('Dice Score')
    ax.set_title('Tooth Segmentation Dice Distribution (Test Set)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/figures/dice_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Show best, median, worst cases
if results:
    results_sorted = sorted(results, key=lambda r: r['dice'])
    worst  = results_sorted[0]
    median = results_sorted[len(results_sorted)//2]
    best   = results_sorted[-1]

    fig, axes = plt.subplots(3, 3, figsize=(12, 12))
    for row, (case, label) in enumerate([
        (best,   f'Best  (Dice={best["dice"]:.3f})'),
        (median, f'Median (Dice={median["dice"]:.3f})'),
        (worst,  f'Worst  (Dice={worst["dice"]:.3f})'),
    ]):
        axes[row,0].imshow(case['image'], cmap='gray')
        axes[row,0].set_title(f'{label}\n{case["image_id"][:12]}', fontsize=8)

        axes[row,1].imshow(case['image'], cmap='gray')
        axes[row,1].imshow(case['gt'], cmap='Greens', alpha=0.5)
        axes[row,1].imshow(case['pred'], cmap='Reds', alpha=0.3)
        axes[row,1].set_title('GT (green) / Pred (red)', fontsize=8)

        axes[row,2].imshow(case['unc_map'], cmap='hot', vmin=0, vmax=1)
        axes[row,2].set_title(f'Uncertainty  (mean={case["uncertainty"]:.3f})', fontsize=8)

        for ax in axes[row]: ax.axis('off')

    plt.suptitle('Best / Median / Worst Cases — Tooth Segmentation', fontsize=11)
    plt.tight_layout()
    plt.savefig('../outputs/figures/best_median_worst_cases.png', dpi=150, bbox_inches='tight')
    plt.show()